<div align="center">
  <img src="Logo_UNIR.png" width="300">
</div>

# “ACTIVIDAD GRUPAL – Uso avanzado de bases de datos NoSQL”
## Nombre y Apellidos: 
### Getulio Cesar De Leon Fernandez 5882984 - 964830
### Oscar Rodrigo Alvarez Urbina 5905388 - 982515
### Andrea Guadalupe Vázquez Moran 5907304 - 976029
### Andrea del Pilar Gonzalez Henriquez 5973251 - 1033152
## **Profesor:**  Barbaro Jorge Ferro Castro
## **Materia:** Bases de Datos para Datos Masivos
## **Maestría:** Maestría en Análisis y Visualización de Datos Masivos
### Universidad Internacional de La Rioja (UNIR)
### Fecha de entrega: 12/01/26
### Grupo:  Equipo 1050E

### Desarollo de la Actividad

Codigo para conectar Python con el Servidor Docker

In [2]:
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017", serverSelectionTimeoutMS=5000)

Codigo para comprobar conexión con python

In [5]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017", serverSelectionTimeoutMS=5000)
client.admin.command("ping")
print("✅ Conectado a MongoDB")
print(client.list_database_names())


✅ Conectado a MongoDB
['Madrid', 'admin', 'config', 'countries', 'local', 'people', 'pruebaDocker']


**2. Importar la base de datos**  
**2.1** Cread la base de datos *inspections* y dentro de ella la colección que corresponda a los datos del fichero *act-grupal-city_inspections.json*:

In [ ]:
# Importa Path para manejar rutas de archivos de forma robusta (Windows/Linux/Mac)
from pathlib import Path

# json_util permite leer JSON “extendido” de MongoDB (por ejemplo $oid, $date, etc.)
from bson import json_util

# BulkWriteError se usa para capturar errores parciales cuando insertamos muchos documentos a la vez
from pymongo.errors import BulkWriteError


# Ruta ABSOLUTA al archivo .json 
p = Path(r"C:\Users\getu9\Desktop\MaestriaCienciasdeDatos\MaestriaUnirDatos\Actividad_2\act-grupal-city_inspections.json")

# Selecciona la base de datos "inspections" y la colección "inspections"
# Nota: 'client' debe estar creado previamente con MongoClient(...)
col = client["inspections"]["inspections"]

# Borra la colección si ya existe, para asegurar que la carga sea repetible (no duplica datos)
col.drop()


# Tamaño del lote (batch): cuántos documentos acumulamos antes de insertarlos en bloque
# Insertar en lotes
batch_size = 2000

# Lista que almacenará temporalmente los documentos antes de insertarlos 
batch = []

# Contador de documentos insertados (para reportar al final)
inserted = 0


# Abre el archivo en modo lectura
# encoding="utf-8-sig" permite leer archivos UTF-8 que incluyen BOM 
with p.open("r", encoding="utf-8-sig") as f:

    # Itera línea por línea (asumiendo formato JSON Lines / JSONL)
    for line in f:

        # Quita espacios en blanco y saltos de línea al inicio/fin
        line = line.strip()

        # Si la línea quedó vacía, se ignora
        if not line:
            continue

        # Convierte la línea (string JSON) a dict de Python
        # IMPORTANTE: json_util.loads entiende tipos extendidos de MongoDB ($oid, $date, etc.)
        doc = json_util.loads(line)

        # Agrega el documento al lote en memoria
        batch.append(doc)

        # Si ya alcanzamos el tamaño del lote, hacemos la inserción masiva
        if len(batch) >= batch_size:
            try:
                # Inserta todos los documentos del lote en una sola operación
                # ordered=False: si un documento falla (ej: duplicado), Mongo sigue insertando los demás
                col.insert_many(batch, ordered=False)

                # Si no hubo errores, asumimos que se insertaron todos los del lote
                inserted += len(batch)

            except BulkWriteError as e:
                # Si hubo errores en algunos documentos, Mongo pudo insertar parcialmente
                # e.details contiene info de cuántos se insertaron realmente
                inserted += e.details.get("nInserted", 0)

            # Limpia el lote para reutilizar la lista y seguir leyendo el archivo
            batch.clear()


# Al terminar el archivo, puede quedar un lote “incompleto” (menos de batch_size)
# Si existe, también hay que insertarlo
if batch:
    try:
        col.insert_many(batch, ordered=False)
        inserted += len(batch)
    except BulkWriteError as e:
        inserted += e.details.get("nInserted", 0)


# Verifica si la base de datos "inspections" aparece en la lista de DBs del servidor
print("✅ DB creada:", "inspections" in client.list_database_names())

# Reporta cuántos documentos se insertaron 
print("✅ Insertados:", inserted)

# Conteo estimado de documentos 
print("✅ Conteo (estimated):", col.estimated_document_count())

# Muestra un documento de ejemplo para validar que la estructura se cargó correctamente
print("✅ Ejemplo:", col.find_one())


✅ DB creada: True
✅ Insertados: 81047
✅ Conteo (estimated): 81047
✅ Ejemplo: {'_id': ObjectId('56d61033a378eccde8a8354f'), 'id': '10021-2015-ENFO', 'certificate_number': 9278806, 'business_name': 'ATLIXCO DELI GROCERY INC.', 'date': 'Feb 20 2015', 'result': 'No Violation Issued', 'sector': 'Cigarette Retail Dealer - 127', 'address': {'city': 'RIDGEWOOD', 'zip': 11385, 'street': 'MENAHAN ST', 'number': 1712}}


El fichero act-grupal-city_inspections.json no está en formato JSON “normal” (un único documento JSON), sino en formato NDJSON / JSON Lines.

No existe un único JSON que contenga todo el contenido, como un array. En cambio, el archivo contiene muchos documentos JSON independientes, 
normalmente uno por línea.

**2.2** Cread la base de datos *countries* y dentro de ella dos colecciones que correspondan con los ficheros *act-grupal-countries-small.json* y *act-grupal-countries-big.json*.


In [ ]:
# Importa Path para manejar rutas de archivos de forma segura y multiplataforma
from pathlib import Path

# Importa json_util para poder leer JSON “extendido” de MongoDB (EJSON),
# por ejemplo: {"_id": {"$oid": "..."}} o fechas {"$date": "..."}
from bson import json_util

# Carpeta base donde están los archivos de la actividad
BASE = Path(r"C:\Users\getu9\Desktop\MaestriaCienciasdeDatos\MaestriaUnirDatos\Actividad_2")

# Ruta del archivo pequeño 
p_small = BASE / "act-grupal-countries-small.json"

# Ruta del archivo grande 
p_big = BASE / "act-grupal-countries-big.json"

# Selecciona/crea la base de datos "countries" en MongoDB
# Nota: 'client' debe estar definido antes (MongoClient(...))
db = client["countries"]

# Selecciona/crea la colección "small" dentro de la DB "countries"
col_small = db["small"]

# Selecciona/crea la colección "big" dentro de la DB "countries"
col_big = db["big"]

# Borra la colección "small" si existía (para que el proceso sea repetible)
col_small.drop()

# Borra la colección "big" si existía (para que el proceso sea repetible)
col_big.drop()

def import_ndjson_ejson(path: Path, col, batch_size=2000):
    """
    Importa un archivo NDJSON (una línea = un documento) que puede contener EJSON de MongoDB,
    y lo inserta en la colección indicada en lotes (batch) para ser más eficiente.
    """

    # Lista temporal donde acumulamos documentos antes de insertarlos
    batch = []

    # Contador de documentos insertados (solo el conteo por nuestro lado)
    inserted = 0

    # Abre el archivo en modo lectura con utf-8-sig (tolera BOM)
    with path.open("r", encoding="utf-8-sig") as f:

        # Recorre cada línea del archivo (cada línea debe ser un JSON válido)
        for line in f:

            # Quita espacios y saltos de línea al inicio y final
            line = line.strip()

            # Si la línea está vacía, la ignora
            if not line:
                continue

            # Convierte la línea (string) a dict de Python,
            # interpretando EJSON ($oid/$date/...) correctamente
            doc = json_util.loads(line)

            # Agrega el documento al lote actual
            batch.append(doc)

            # Si el lote alcanzó el tamaño definido, inserta en MongoDB
            if len(batch) >= batch_size:

                # Inserta todos los docs del lote (ordered=False continúa aunque uno falle)
                col.insert_many(batch, ordered=False)

                # Actualiza el contador de insertados
                inserted += len(batch)

                # Vacía el lote para empezar a acumular otro
                batch.clear()

    # Al terminar el archivo, si quedó un lote incompleto, también se inserta
    if batch:
        col.insert_many(batch, ordered=False)
        inserted += len(batch)

    # Devuelve cuántos documentos insertamos (según nuestro conteo)
    return inserted

# Importa el dataset pequeño en la colección "small"
ins_small = import_ndjson_ejson(p_small, col_small)

# Importa el dataset grande en la colección "big"
ins_big = import_ndjson_ejson(p_big, col_big)

# Verifica si la DB "countries" aparece en la lista de bases existentes
print("✅ DB creada:", "countries" in client.list_database_names())

# Muestra insertados y conteo estimado en MongoDB para "small"
print("✅ Insertados small:", ins_small, " count:", col_small.estimated_document_count())

# Muestra insertados y conteo estimado en MongoDB para "big"
print("✅ Insertados big:", ins_big, " count:", col_big.estimated_document_count())

# Muestra un documento de ejemplo de la colección "small"
print("✅ small sample:", col_small.find_one())

# Muestra un documento de ejemplo de la colección "big"
print("✅ big sample:", col_big.find_one())


✅ DB creada: True
✅ Insertados small: 248  count: 248
✅ Insertados big: 21640  count: 21640
✅ small sample: {'_id': ObjectId('55a0f42f20a4d760b5fc305e'), 'altSpellings': ['AI'], 'area': 91, 'borders': [], 'callingCode': ['1264'], 'capital': 'The Valley', 'cca2': 'AI', 'cca3': 'AIA', 'ccn3': '660', 'cioc': '', 'currency': ['XCD'], 'demonym': 'Anguillian', 'landlocked': False, 'languages': {'eng': 'English'}, 'latlng': [18.25, -63.16666666], 'name': {'common': 'Anguilla', 'native': {'eng': {'common': 'Anguilla', 'official': 'Anguilla'}}, 'official': 'Anguilla'}, 'region': 'Americas', 'subregion': 'Caribbean', 'tld': ['.ai'], 'translations': {'deu': {'common': 'Anguilla', 'official': 'Anguilla'}, 'fin': {'common': 'Anguilla', 'official': 'Anguilla'}, 'fra': {'common': 'Anguilla', 'official': 'Anguilla'}, 'hrv': {'common': 'Angvila', 'official': 'Anguilla'}, 'ita': {'common': 'Anguilla', 'official': 'Anguilla'}, 'jpn': {'common': 'アンギラ', 'official': 'アングィラ'}, 'nld': {'common': 'Anguilla', 

Estos dos tambien no estan, en formato JSON “normal” (un único documento JSON), sino en formato NDJSON / JSON Lines.

**4. Restaurar una base de datos:**  

**4.1** Para restaurar una base de datos, debéis indicar la ruta del directorio dump donde se almacenan los ficheros de la base de datos (json y bson).
dump
(1)	people
(2)	--people.bson
(3)	--people.metadata.json

**4.2** Descomprimid el fichero act-grupal-people.zip y ubicad los ficheros en la estructura de directorios indicada en el paso anterior

Verificamos que se haya restaurado correctamente:

In [128]:
print(client["people"].list_collection_names())
print(client["people"]["people"].count_documents({}))


['people']
1000000


**5. Caso de uso: restricción de terrazas en Madrid por covid-19.**  
Contratan a vuestro equipo para actualizar las restricciones de ciertos locales y terrazas en Madrid por cuestiones del covid-19. Los datos para actualizar están en el fichero llamado act-grupal-openDataLocalesMadrid.cvs y se os pide que consolidéis dichos cambios en una base de datos MongoDB llamada Madrid con la colección Terrazas. La información del dataset a utilizar lo tenéis en la siguiente URL: Open Data Censo de locales, sus actividades y terrazas de hostelería y restauración (Terrazas)



In [ ]:
from pathlib import Path              # Importa Path para manejar rutas de archivos de forma segura (Windows/Linux)
import pandas as pd                   # Importa pandas para leer y analizar el CSV como DataFrame

# 1) Definimos la carpeta base donde están los ficheros de la actividad
BASE = Path(r"C:\Users\getu9\Desktop\MaestriaCienciasdeDatos\MaestriaUnirDatos\Actividad_2")

# 2) Buscamos el archivo cuyo nombre empieza por "act-grupal-openDataLocalesMadrid"
#    glob() devuelve una lista de coincidencias; tomamos la primera [0]
p = list(BASE.glob("act-grupal-openDataLocalesMadrid*"))[0]

# 3) Leemos el archivo como texto para poder inspeccionar SOLO la primera línea 
#    - encoding="utf-8-sig" maneja el BOM típico de CSVs en Windows
#    - errors="replace" evita que falle si hay caracteres raros (los reemplaza)
first_line = p.read_text(encoding="utf-8-sig", errors="replace").splitlines()[0]

# 4) Detectamos el separador del archivo mirando la primera línea:
#    - si contiene "|" usamos "|"
#    - si no, si contiene ";" usamos ";"
#    - si no, asumimos ","
sep = "|" if "|" in first_line else (";" if ";" in first_line else ",")

# 5) Leemos el archivo con pandas usando el separador detectado:
#    - dtype=str: fuerza todas las columnas a texto para no perder ceros a la izquierda
#      y evitar conversiones automáticas 
#    - encoding="utf-8-sig": mantiene consistencia con la lectura anterior
df = pd.read_csv(p, sep=sep, encoding="utf-8-sig", dtype=str)

# 6) Limpiamos los nombres de columnas quitando espacios (por ejemplo " nombre " -> "nombre")
df.columns = [c.strip() for c in df.columns]

# 7) Mostramos un resumen rápido del archivo para verificar que se leyó correctamente
print("✅ Archivo:", p.name)                                 # Nombre del archivo encontrado
print("✅ Separador:", repr(sep))                            # Separador detectado (repr para verlo claro)
print("✅ Filas:", len(df), "| Columnas:", len(df.columns))  # Tamaño del dataset (filas y columnas)
print("✅ Columnas (primeras 20):", df.columns.tolist()[:20])# Lista de columnas (solo primeras 20)


✅ Archivo: act-grupal-openDataLocalesMadrid.csv
✅ Separador: ';'
✅ Filas: 2999 | Columnas: 44
✅ Columnas (primeras 20): ['_id', 'id_local', 'id_distrito_local', 'desc_distrito_local', 'desc_barrio_local', 'clase_vial_edificio', 'num_edificio', 'Cod_Postal', 'coordenada_x_local', 'coordenada_y_local', 'desc_situacion_local', 'Nombre', 'id_periodo_terraza', 'desc_periodo_terraza', 'id_situacion_terraza', 'desc_situacion_terraza', 'Superficie_ES', 'DESC_CLASE', 'DESC_NOMBRE', 'num_terraza']


Se creó la base de datos Madrid y la colección Terrazas a partir del fichero act-grupal-openDataLocalesMadrid.

In [ ]:
from datetime import datetime, timezone  # Importa datetime para generar timestamps; timezone.utc para guardar hora en UTC

# Asegura que existe la columna _id
if "_id" not in df.columns:  # Verifica que el DataFrame (df) tenga una columna llamada "_id"
    raise ValueError(        # Si no existe, lanza un error explicando el problema
        "El CSV no trae columna '_id'. Dime qué columna quieres usar como identificador (ej: id_terraza)."
    )

# limpia _id
df["_id"] = df["_id"].astype(str).str.strip()  # Convierte _id a string y elimina espacios al inicio/fin (normaliza el identificador)
df2 = df[df["_id"].notna() & (df["_id"] != "") & (df["_id"].str.lower() != "nan")].copy()
# Filtra filas con _id válido: no nulo, no vacío, y no el string "nan"; luego hace copy() para trabajar seguro sin SettingWithCopyWarning

df2 = df2.drop_duplicates(subset=["_id"], keep="last")
# Si hay IDs duplicados, se queda con la última aparición (keep="last") para tener una única fila por _id

now = datetime.now(timezone.utc)  # Captura un timestamp único (UTC) para marcar la actualización de TODOS los docs en esta corrida

ops = []  # Lista donde guardaremos las operaciones tipo UpdateOne para hacer un bulk_write (muchas actualizaciones en una sola llamada)
for row in df2.to_dict(orient="records"):  # Recorre el DataFrame como lista de diccionarios (1 dict por fila)
    mongo_id = str(row["_id"]).strip()     # Toma el _id de la fila y lo normaliza (string sin espacios)

    # doc limpio: sin NaN/vacíos
    doc = {k: v for k, v in row.items()
           if v is not None and str(v).strip() != "" and str(v).lower() != "nan"}
    # Construye un diccionario "doc" con SOLO los campos cuyo valor sea útil:
    # - no sea None
    # - no sea vacío (tras strip)
    # - no sea "nan" (cuando pandas lo trajo como string)
  

    doc.pop("_id", None)     # Elimina _id del dict porque no se debe actualizar el _id con $set (Mongo no permite cambiarlo)
    doc["_updated_at"] = now # Añade campo de auditoría: cuándo se actualizó
    doc["_source_file"] = p.name  # Añade de dónde viene la data: nombre del archivo procesado (p es la ruta del CSV)

    ops.append(  # Agrega una operación UpdateOne a la lista para ejecutarla luego en bloque
        UpdateOne(  # Tipo de operación: update de un documento
            {"_id": mongo_id},  # Filtro: el documento objetivo es el que tenga este _id
            {
                "$set": doc,  # Actualiza/crea estos campos con los valores limpios
                "$setOnInsert": {"_id": mongo_id}  # Si no existe y se inserta, fija el _id una sola vez
            },
            upsert=True  # Upsert=True significa: si no existe el _id, lo inserta; si existe, lo actualiza
        )
    )

try:
    res = col.bulk_write(ops, ordered=False)  # Ejecuta todas las operaciones juntas; ordered=False permite continuar aunque una falle
    print("✅ Carga OK")                      # Mensaje de éxito si no explota el bloque
    print("Matched:", res.matched_count)      # Cuántos docs encontraron match (existían) para actualizar
    print("Modified:", res.modified_count)    # Cuántos docs realmente cambiaron (puede ser menor que matched)
    print("Upserted:", len(res.upserted_ids)) # Cuántos docs fueron insertados nuevos (upserts)
    print("Total docs:", col.estimated_document_count())  # Conteo aproximado total en la colección
    print("Sample:", col.find_one({}, {"_id": 1, "id_terraza": 1, "id_local": 1}))
    # Muestra un documento de ejemplo, proyectando solo _id, id_terraza e id_local para verificar rápidamente

except BulkWriteError as e:
    print("❌ Primer error:", e.details["writeErrors"][0])
    # Si hay errores en el bulk (por ejemplo duplicados, tipos inválidos, etc.),
    # imprime el primer error para diagnosticar rápido


✅ Carga OK
Matched: 0
Modified: 0
Upserted: 2999
Total docs: 5998
Sample: {'_id': 7, 'id_local': 280067128}


In [205]:
print("DBs:", client.list_database_names())
print("Colecciones en Madrid:", db.list_collection_names())
print("Docs en Terrazas:", col.count_documents({}))


DBs: ['Madrid', 'admin', 'config', 'countries', 'inspections', 'local', 'people', 'pruebaDocker']
Colecciones en Madrid: ['Zona1', 'Zona2', 'Terrazas']
Docs en Terrazas: 5998


Se realizó una carga por upsert usando el campo _id del dataset como identificador del documento para permitir consolidación (si se ejecuta de nuevo, actualizaría en vez de duplicar).
Resultado: 2999 documentos cargados correctamente en Madrid.Terrazas.

- En el alcance del contrato se os pide:

  **5.1** Convertir el fichero CSV a formato JSON. Utilizad, por ejemplo, el servicio de  
     https://csvjson.com/csv2json

  **5.2** Descargar el fichero generado en el paso anterior, en formato JSON indicando la opción **Array**
     (todos los documentos estarán dentro de un único *array*). En el fichero hay un problema de espacios
     en muchos campos String, intentad corregirlo en esta etapa.

  **5.3** Editar el fichero generado, y al principio del todo añadir la línea:

     ```js
     var datos_insertar = [...]
     ```

  **5.4** Guardar el fichero en una ruta conocida y cambiad su extensión a formato **.js**.  
     ¿Sabéis qué tipo de fichero es un **.js**?

  **5.5** Abrir el cliente *mongo* y ejecutar las siguientes instrucciones:

     ```js
     load("PATH\\fichero.js")
     datos_insertar[0]
     ```


  **5.6** Describir brevemente qué hacen las instrucciones anteriores.
  
  **Respuesta:** *load("PATH\\fichero.js")*: carga y ejecuta el archivo .js dentro de la Mongo Shell. Como ese archivo empieza con var datos_insertar = [...], al ejecutarlo crea en memoria la variable datos_insertar (un array con todos los documentos del JSON).
  *datos_insertar[0]*: accede al primer elemento de ese array y lo muestra en pantalla, para verificar que los datos se cargaron bien y ver la estructura de un documento.

  **5.7** Con el elemento **datos_insertar**, ¿podríais realizar un *insert* masivo? ¿Cómo?

  **Respuesta:** utilizamos el siguiente codigo

   use Madrid  
  db.Terrazas.insertMany(datos_insertar, { ordered: false })

  **5.8** Desde la terminal, podríais proponer una alternativa que evite realizar los pasos anteriores
     (descripción de dos líneas máximo y no vale indicar el uso MongoCompass 😊).

  **Respuesta:** utilizamos el siguiente codigo
  
  mongoimport --uri "mongodb://localhost:27017" `
  --db Madrid `  
  --collection Terrazas `  
  --file "C:\Users\getu9\Desktop\MaestriaCienciasdeDatos\MaestriaUnirDatos\Actividad_2\act-grupal-openDataLocalesMadrid_trim.json" `  
  --jsonArray `  
  --drop



In [ ]:
import json                    # Librería estándar para leer/escribir JSON
from pathlib import Path       # Path para manejar rutas de archivos de forma robusta

# Ruta del archivo de entrada: JSON en formato ARRAY (todos los documentos dentro de un único array)
inp = Path(r"C:\Users\getu9\Desktop\MaestriaCienciasdeDatos\MaestriaUnirDatos\Actividad_2\act-grupal-openDataLocalesMadrid.json")

# Ruta del archivo de salida: versión "limpia" donde los strings se recortan (trim)
out = Path(r"C:\Users\getu9\Desktop\MaestriaCienciasdeDatos\MaestriaUnirDatos\Actividad_2\act-grupal-openDataLocalesMadrid_trim.json")

# Lee todo el archivo JSON como texto (UTF-8) y lo convierte a objetos Python (list/dict/str/etc.)
data = json.loads(inp.read_text(encoding="utf-8"))

def trim_strings(x):
    """
    Función recursiva para limpiar strings dentro de estructuras complejas:
    - Si x es string -> aplica strip() (quita espacios al inicio y final)
    - Si x es lista  -> limpia cada elemento
    - Si x es dict   -> limpia cada valor (y preserva las claves)
    - Si es otro tipo (int/float/bool/None) -> lo devuelve igual
    """
    if isinstance(x, str):             # Caso 1: el valor es un string
        return x.strip()               # Quita espacios al inicio y al final

    if isinstance(x, list):            # Caso 2: el valor es una lista (array)
        return [trim_strings(i) for i in x]   # Limpia cada elemento de la lista

    if isinstance(x, dict):            # Caso 3: el valor es un diccionario (objeto JSON)
        return {k: trim_strings(v) for k, v in x.items()}  # Limpia cada valor del diccionario

    return x                           # Caso 4: otros tipos (números, booleanos, None) se dejan igual


# Aplica la limpieza a TODA la estructura (lista de documentos) de forma recursiva
data2 = trim_strings(data)

# Convierte el resultado de vuelta a JSON:
# - indent=2 lo deja legible para revisión/entrega
out.write_text(json.dumps(data2, ensure_ascii=False, indent=2), encoding="utf-8")

# Mensaje final para confirmar que se generó el archivo limpio
print("OK ->", out)


OK -> C:\Users\getu9\Desktop\MaestriaCienciasdeDatos\MaestriaUnirDatos\Actividad_2\act-grupal-openDataLocalesMadrid_trim.json


En el fichero hay un problema de espacios en muchos campos String, intentad corregirlo en esta etapa. 


Editar el fichero generado, y al principio del todo añadir la línea:
var datos_insertar = […]
d. Guardar el fichero en una ruta conocida y cambiad su extensión a formato .js. 


In [292]:
from pathlib import Path  # Importa Path para manejar rutas de archivos de forma segura y clara

# 1) Define la carpeta base donde están los archivos del laboratorio
base = Path(r"C:\Users\getu9\Desktop\MaestriaCienciasdeDatos\MaestriaUnirDatos\Actividad_2")

# 2) Ruta del archivo de entrada: JSON ya corregido (formato ARRAY: [ {...}, {...} ])
inp = base / "act-grupal-openDataLocalesMadrid_trim.json"

# 3) Ruta del archivo de salida: lo convertiremos a .js para poder usarlo con load() en mongosh
out = base / "act-grupal-openDataLocalesMadrid_trim.js"

# 4) Lee todo el contenido del JSON como texto (se mantiene exactamente el array JSON)
json_text = inp.read_text(encoding="utf-8")

# 5) Crea el archivo .js añadiendo al inicio:
#    "var datos_insertar = " + (el array JSON) + ";"
#    Esto convierte el JSON en un script JavaScript válido que define una variable
#    llamada datos_insertar con todos los documentos dentro.
out.write_text("var datos_insertar = " + json_text + ";\n", encoding="utf-8")

# 6) Imprime confirmación de que el archivo .js fue creado y dónde quedó guardado
print("✅ Creado:", out)


✅ Creado: C:\Users\getu9\Desktop\MaestriaCienciasdeDatos\MaestriaUnirDatos\Actividad_2\act-grupal-openDataLocalesMadrid_trim.js


¿Sabéis qué tipo de fichero es un .js?  

**Respuesta:** Un archivo .js es un archivo de código JavaScript (un script).  es JavaScript puro, y Mongo Shell puede ejecutar JavaScript

Realizad las siguientes actualizaciones sobre los datos (cuando los cambios afecten a campos _es o _ra, haced el update sobre los datos del local en período estacional):


**6.1** Los locales del barrio Guindalera de Salamanca, por motivos de la desescalada, no podrán abrir y tendrán que permanecer cerrados. Cambiad la situación del local a Cerrado utilizando el id correspondiente. Recordad que debéis constatar los tipos de situaciones que existan y seguir con la misma codificación. A estos mismos locales, cambiad la situación de la terraza a cerrada siguiendo las mismas premisas anteriores.

In [ ]:
from pymongo import MongoClient  # Importa el cliente de MongoDB para conectar y operar con la base de datos

client = MongoClient("mongodb://localhost:27017", serverSelectionTimeoutMS=5000)  # Crea conexión a MongoDB local (timeout 5s)
client.admin.command("ping")  # Verifica conectividad 
print("✅ Conectado a MongoDB")  # Mensaje de confirmación de conexión

col = client["Madrid"]["Terrazas"]  # Selecciona la base de datos "Madrid" y la colección "Terrazas"

def norm(s):
    """Normaliza strings para comparar: convierte a string, recorta espacios y pasa a minúsculas."""
    return str(s).strip().lower()  # Limpia espacios y unifica en minúsculas

def codificacion(col, id_field, desc_field, title):
    """
    Obtiene la 'codificación' real existente en los datos:
    agrupa por (id, descripción) y cuenta cuántos documentos tienen esa combinación.
    """
    pipeline = [
        # Agrupa documentos por la combinación (id_field, desc_field) y cuenta cuántos hay en cada grupo
        {"$group": {"_id": {"id": f"${id_field}", "desc": f"${desc_field}"}, "n": {"$sum": 1}}},
        # Reestructura el resultado para dejar columnas limpias: id, desc, n
        {"$project": {"_id": 0, "id": "$_id.id", "desc": "$_id.desc", "n": 1}},
        # Ordena por frecuencia descendente (las combinaciones más comunes arriba)
        {"$sort": {"n": -1}}
    ]

    rows = list(col.aggregate(pipeline))  # Ejecuta el pipeline de agregación y guarda todos los resultados en una lista
    print(f"\n✅ Codificación {title} ({id_field} -> {desc_field}):")  # Encabezado informativo para el reporte/console

    # Imprime todas las combinaciones (id -> descripción) con su conteo
    for r in rows:
        print(f"  {r.get('id')} -> {r.get('desc')} | n={r.get('n')}")

    return rows  # Devuelve el listado de codificaciones encontradas

def try_resolver(rows, desired_desc):
    """
    Busca dentro de 'rows' (codificaciones) una descripción deseada:
    1) match exacto (normalizado)
    2) match parcial (la deseada contenida dentro del texto)
    Devuelve (id, desc) o (None, None) si no encuentra.
    """
    # Primero intenta encontrar coincidencia EXACTA (ignorando mayúsculas/espacios)
    for r in rows:
        if norm(r.get("desc")) == norm(desired_desc):
            return r.get("id"), r.get("desc")  # Devuelve el id y la descripción real del dataset

    # Si no hay match exacto, intenta coincidencia PARCIAL (contiene)
    for r in rows:
        if norm(desired_desc) in norm(r.get("desc", "")):
            return r.get("id"), r.get("desc")  # Devuelve el id y la descripción que más se aproxima

    return None, None  # Si no hay coincidencias, devuelve None

# 1) Leer codificaciones reales (requisito del enunciado: constatar los tipos existentes y su codificación)
local_rows = codificacion(col, "id_situacion_local", "desc_situacion_local", "LOCAL")  # Codificación para situación del LOCAL
terraza_rows = codificacion(col, "id_situacion_terraza", "desc_situacion_terraza", "TERRAZA")  # Codificación para situación de la TERRAZA

# 2) Resolver "Cerrado" para LOCAL (debe existir en el dataset)
id_local_cerrado, desc_local_cerrado = try_resolver(local_rows, "Cerrado")  # Busca el id asociado a "Cerrado" en locales
if desc_local_cerrado is None:
    raise ValueError("No existe 'Cerrado' en desc_situacion_local.")  # Si no existe, detiene el script: no se puede inventar codificación

# 3) Resolver "Cerrada" para TERRAZA (si no existe, se usa fallback razonable: "Suspension temporal")
id_terr, desc_terr = try_resolver(terraza_rows, "Cerrada")  # Intenta encontrar "Cerrada" como estado de terraza
if desc_terr is None:
    id_terr, desc_terr = try_resolver(terraza_rows, "Suspension temporal")  # Si no existe "Cerrada", usa "Suspension temporal"

if desc_terr is None:
    raise ValueError("No existe 'Cerrada' NI 'Suspension temporal' en desc_situacion_terraza.")  # Si no existe ninguno, detiene el script

print("\n✅ Usaré estos valores:")  # Muestra qué codificaciones se van a aplicar
print("  LOCAL   :", id_local_cerrado, "->", desc_local_cerrado)  # Codificación final elegida para el LOCAL
print("  TERRAZA :", id_terr,          "->", desc_terr)          # Codificación final elegida para la TERRAZA

# 4) Filtro: seleccionar los documentos que pertenezcan al distrito Salamanca y barrio Guindalera (case-insensitive)
filtro = {
    "desc_distrito_local": {"$regex": r"^SALAMANCA$", "$options": "i"},  # Distrito Salamanca (exacto, ignorando mayúsculas/minúsculas)
    "desc_barrio_local": {"$regex": r"^GUINDALERA$", "$options": "i"},   # Barrio Guindalera (exacto, ignorando mayúsculas/minúsculas)
}

print("\n📌 Docs afectados:", col.count_documents(filtro))  # Cuenta cuántos documentos cumplen el filtro (para evidencia)

# 5) Updates REPETIBLES (idempotentes):
#    construye los $set con el texto y, si existe id, también actualiza el id correspondiente.
set_local = {"desc_situacion_local": desc_local_cerrado}  # Cambia la descripción de situación del local
if id_local_cerrado is not None:
    set_local["id_situacion_local"] = id_local_cerrado  # Actualiza también el id (si existe en el dataset; si es None, no fuerza el campo)

set_terr = {"desc_situacion_terraza": desc_terr}  # Cambia la descripción de situación de la terraza
if id_terr is not None:
    set_terr["id_situacion_terraza"] = id_terr  # Actualiza también el id de situación de terraza (si existe)

r1 = col.update_many(filtro, {"$set": set_local})  # Aplica update masivo a todos los docs que cumplen el filtro (situación del local)
r2 = col.update_many(filtro, {"$set": set_terr})   # Aplica update masivo a todos los docs que cumplen el filtro (situación de la terraza)

print("\n✅ UPDATE LOCAL  :", "matched", r1.matched_count, "modified", r1.modified_count)  # Evidencia: cuántos coincidieron y cuántos cambiaron
print("✅ UPDATE TERRAZA:", "matched", r2.matched_count, "modified", r2.modified_count)    # Evidencia: cuántos coincidieron y cuántos cambiaron

# 6) Muestra: imprime un documento de ejemplo para comprobar el resultado final del update
sample = col.find_one(
    filtro,  # Busca cualquier doc que cumpla el filtro (post-update)
    {
        "_id": 1, "id_local": 1,  # Campos de identificación
        "desc_distrito_local": 1, "desc_barrio_local": 1,  # Contexto geográfico
        "id_situacion_local": 1, "desc_situacion_local": 1,  # Estado del local
        "id_situacion_terraza": 1, "desc_situacion_terraza": 1  # Estado de la terraza
    }
)
print("\n🔎 Ejemplo post-update:", sample)  # Muestra el ejemplo para validar que quedó “Cerrado” / “Suspension temporal” (o “Cerrada” si existía)


✅ Conectado a MongoDB

✅ Codificación LOCAL (id_situacion_local -> desc_situacion_local):
  None -> Abierto | n=2931
  None -> Cerrado | n=63
  None -> Baja Reunificaci? | n=5

✅ Codificación TERRAZA (id_situacion_terraza -> desc_situacion_terraza):
  1 -> Abierta | n=2953
  8 -> Suspension temporal | n=46

✅ Usaré estos valores:
  LOCAL   : None -> Cerrado
  TERRAZA : 8 -> Suspension temporal

📌 Docs afectados: 32

✅ UPDATE LOCAL  : matched 32 modified 0
✅ UPDATE TERRAZA: matched 32 modified 0

🔎 Ejemplo post-update: {'_id': 33, 'id_local': 270403150, 'desc_distrito_local': 'SALAMANCA', 'desc_barrio_local': 'GUINDALERA', 'desc_situacion_local': 'Cerrado', 'id_situacion_terraza': 8, 'desc_situacion_terraza': 'Suspension temporal'}


LOCAL: todos tienen id_situacion_local = None (o sea, no hay codificación numérica real cargada ahí), pero sí existe desc_situacion_local con “Cerrado”.

TERRAZA: sí hay codificación numérica (1 y 8), pero no existe “Cerrada”. El equivalente “de cierre” es “Suspension temporal”.

**6.2** A todas las terrazas que se ubiquen en la acera, añadid un campo llamado inspeccionar y estableced el siguiente valor:
Si dispone de más de 10 mesas, true.
Si dispone de menos de 10 mesas, false.


In [504]:
from pymongo import MongoClient

# =========================
# CONEXIÓN A MONGODB
# =========================

# Crea un cliente de MongoDB apuntando al servidor local (timeout corto para fallar rápido si no conecta)
client = MongoClient("mongodb://localhost:27017", serverSelectionTimeoutMS=5000)

# Fuerza un ping para validar que la conexión realmente funciona
client.admin.command("ping")

# Selecciona la colección Madrid.Terrazas
col = client["Madrid"]["Terrazas"]

# Mensaje informativo
print("✅ Conectado a MongoDB y colección Madrid.Terrazas")

# =========================
# FILTROS (ACERA + ESTACIONAL)
# =========================

# Filtro 1: SOLO terrazas que están ubicadas exactamente en "Acera"
# (esto evita que entren valores como "Bulevar" u otros y elimina los falsos positivos)
filtro_acera = {
    "desc_ubicacion_terraza": {"$regex": r"^\s*acera\s*$", "$options": "i"}
}

# Filtro 2: SOLO período estacional (según tu dataset, id_periodo_terraza=2 suele ser "Estacional")
# Incluimos también la opción por texto por si algunos registros vienen sin id pero con descripción
filtro_estacional = {
    "$or": [
        {"id_periodo_terraza": 2},
        {"desc_periodo_terraza": {"$regex": r"^\s*estacional\s*$", "$options": "i"}}
    ]
}

# Filtro final: Acera Y Estacional
filtro = {"$and": [filtro_acera, filtro_estacional]}

# =========================
# CONFIG DEL CAMPO DE MESAS
# =========================

# Campo que vamos a usar para la regla:
# - El enunciado pide que si afecta a _es o _ra, se haga update sobre el período estacional.
# - Por eso aquí usamos mesas_es.
mesas_field = "mesas_es"

# =========================
# UPDATE MASIVO (PIPELINE UPDATE)
# =========================

# Pipeline de update:
# - Convierte mesas_es a entero de forma segura (si viene vacío / letras -> 0)
# - Crea/actualiza el campo "inspeccionar" con la regla:
#     True  si mesas > 10
#     False si mesas <= 10
pipeline = [
    {
        "$set": {
            "inspeccionar": {
                "$gt": [
                    {
                        "$convert": {
                            "input": {"$ifNull": [f"${mesas_field}", "0"]},  # si mesas_es es null -> "0"
                            "to": "int",                                     # convierte a entero
                            "onError": 0,                                    # si falla la conversión -> 0
                            "onNull": 0                                      # si es null -> 0
                        }
                    },
                    10  # umbral: "más de 10" => True
                ]
            }
        }
    }
]

# Cuenta cuántos documentos serán afectados antes del update (para reportarlo)
total_objetivo = col.count_documents(filtro)

# Ejecuta el update masivo con pipeline (MongoDB 4.2+)
res = col.update_many(filtro, pipeline)

# Reporta resultados del update
print(f"📌 Total acera+estacional (antes): {total_objetivo}")
print(f"✅ Matched: {res.matched_count} | Modified: {res.modified_count}")

# =========================
# VERIFICACIÓN (CONTEOS + ERRORES)
# =========================

# Cuenta cuántos quedaron con inspeccionar True
n_true = col.count_documents({**filtro, "inspeccionar": True})

# Cuenta cuántos quedaron con inspeccionar False
n_false = col.count_documents({**filtro, "inspeccionar": False})

# Cuenta cuántos NO tienen el campo inspeccionar (idealmente debe ser 0)
n_missing = col.count_documents({**filtro, "inspeccionar": {"$exists": False}})

print("\n📊 Verificación básica:")
print("True :", n_true)
print("False:", n_false)
print("Sin campo inspeccionar:", n_missing)

# Verifica si hay errores lógicos:
# 1) inspeccionar=True pero mesas_es <= 10  -> ERROR
errores_true_mesas_le_10 = col.count_documents({
    **filtro,
    "inspeccionar": True,
    "$expr": {
        "$lte": [
            {
                "$convert": {
                    "input": {"$ifNull": [f"${mesas_field}", "0"]},
                    "to": "int",
                    "onError": 0,
                    "onNull": 0
                }
            },
            10
        ]
    }
})

# 2) inspeccionar=False pero mesas_es > 10 -> ERROR
errores_false_mesas_gt_10 = col.count_documents({
    **filtro,
    "inspeccionar": False,
    "$expr": {
        "$gt": [
            {
                "$convert": {
                    "input": {"$ifNull": [f"${mesas_field}", "0"]},
                    "to": "int",
                    "onError": 0,
                    "onNull": 0
                }
            },
            10
        ]
    }
})

print("\n🧪 Chequeo de consistencia (regla >10):")
print("❌ Errores (True con mesas<=10):", errores_true_mesas_le_10)
print("❌ Errores (False con mesas>10):", errores_false_mesas_gt_10)

# =========================
# MUESTRAS (EJEMPLOS)
# =========================

print("\n🔎 Ejemplos inspeccionar=True (deberían tener mesas_es > 10):")
for d in col.find(
    {**filtro, "inspeccionar": True},
    {"_id": 1, "id_local": 1, "desc_ubicacion_terraza": 1, mesas_field: 1, "inspeccionar": 1}
).limit(5):
    print(d)

print("\n🔎 Ejemplos inspeccionar=False (deberían tener mesas_es <= 10):")
for d in col.find(
    {**filtro, "inspeccionar": False},
    {"_id": 1, "id_local": 1, "desc_ubicacion_terraza": 1, mesas_field: 1, "inspeccionar": 1}
).limit(5):
    print(d)



✅ Conectado a MongoDB y colección Madrid.Terrazas
📌 Total acera+estacional (antes): 741
✅ Matched: 741 | Modified: 15

📊 Verificación básica:
True : 161
False: 580
Sin campo inspeccionar: 0

🧪 Chequeo de consistencia (regla >10):
❌ Errores (True con mesas<=10): 0
❌ Errores (False con mesas>10): 0

🔎 Ejemplos inspeccionar=True (deberían tener mesas_es > 10):
{'_id': 51, 'id_local': 270076142, 'desc_ubicacion_terraza': 'Acera', 'mesas_es': 14, 'inspeccionar': True}
{'_id': 78, 'id_local': 170001098, 'desc_ubicacion_terraza': 'Acera', 'mesas_es': 11, 'inspeccionar': True}
{'_id': 111, 'id_local': 270023348, 'desc_ubicacion_terraza': 'Acera', 'mesas_es': 30, 'inspeccionar': True}
{'_id': 147, 'id_local': 170000809, 'desc_ubicacion_terraza': 'Acera', 'mesas_es': 30, 'inspeccionar': True}
{'_id': 156, 'id_local': 270363433, 'desc_ubicacion_terraza': 'Acera', 'mesas_es': 12, 'inspeccionar': True}

🔎 Ejemplos inspeccionar=False (deberían tener mesas_es <= 10):
{'_id': 7, 'id_local': 280067128,

**6.3** A las terrazas que se deban inspeccionar, añadid 2 mesas más auxiliares y 8 sillas más disponibles (tanto en período estacional como el resto del año).

In [608]:
from pymongo import MongoClient                     # Importa el cliente de MongoDB para conectarse a la base de datos
from datetime import datetime, timezone             # Importa utilidades de fecha/hora con zona horaria (UTC)

client = MongoClient("mongodb://localhost:27017", serverSelectionTimeoutMS=5000)  # Crea conexión a MongoDB (timeout 5s)
client.admin.command("ping")                        # Verifica que el servidor responde (si falla, lanza error)
col = client["Madrid"]["Terrazas"]                  # Selecciona la colección Madrid.Terrazas

now = datetime.now(timezone.utc)                    # Guarda la fecha/hora actual en UTC para registrar el update

# Marca para que sea repetible (evita aplicar dos veces)
MARK_ES = "_upd_63_es_v1"                           # Campo “bandera” que indica que ya aplicamos el update ESTACIONAL
MARK_RA = "_upd_63_ra_v1"                           # Campo “bandera” que indica que ya aplicamos el update RESTO AÑO

# ---------
# 1) Update período ESTACIONAL (afecta *_es)
# ---------
filtro_es = {                                       # Filtro para seleccionar SOLO los documentos a actualizar (estacional)
    "inspeccionar": True,                           # Solo terrazas que deben ser inspeccionadas
    "$or": [                                        # Condición para identificar el período estacional por id o por texto
        {"id_periodo_terraza": 2},                  # Caso 1: codificación del dataset (2 = Estacional)
        {"desc_periodo_terraza": {"$regex": r"^estacional$", "$options": "i"}}  # Caso 2: texto "estacional" (ignora mayúsculas)
    ],
    MARK_ES: {"$ne": True}                          # Evita volver a aplicar el mismo update si ya se aplicó antes
}

pipeline_es = [{                                   # Pipeline de actualización (estilo “aggregation update”)
    "$set": {                                       # $set recalcula/crea campos en el documento
        # +2 mesas auxiliares estacionales (campo nuevo)
        "mesas_aux_es": {                           # Campo nuevo o existente: mesas auxiliares en período estacional
            "$add": [                               # Suma 2 al valor actual
                {"$convert": {                      # Convierte el valor actual a entero de forma segura
                    "input": "$mesas_aux_es",       # Lee el valor actual del documento
                    "to": "int",                    # Lo convierte a int
                    "onError": 0,                   # Si hay error (texto, etc.) usa 0
                    "onNull": 0                     # Si es null, usa 0
                }},
                2                                   # Incremento: +2 mesas auxiliares
            ]
        },
        # +8 sillas estacionales
        "sillas_es": {                              # Campo existente: sillas en período estacional
            "$add": [                               # Suma 8 al valor actual
                {"$convert": {                      # Convierte el valor actual a entero de forma segura
                    "input": "$sillas_es",          # Lee el valor actual del documento
                    "to": "int",                    # Lo convierte a int
                    "onError": 0,                   # Si hay error, usa 0
                    "onNull": 0                     # Si es null, usa 0
                }},
                8                                   # Incremento: +8 sillas
            ]
        },
        MARK_ES: True,                              # Guarda la marca para que este update sea repetible (no se aplique dos veces)
        "_updated_at": now                          # Guarda timestamp del cambio
    }
}]

res_es = col.update_many(filtro_es, pipeline_es)    # Ejecuta el update masivo para período ESTACIONAL

# ---------
# 2) Update RESTO DEL AÑO (afecta *_ra)
# ---------
filtro_ra = {                                       # Filtro para seleccionar SOLO los documentos a actualizar (resto del año)
    "inspeccionar": True,                           # Solo terrazas que deben ser inspeccionadas
    "$or": [                                        # Condición para identificar “resto del año” por exclusión del estacional
        {"id_periodo_terraza": {"$ne": 2}},         # Caso 1: id_periodo_terraza distinto de 2
        {"desc_periodo_terraza": {"$not": {         # Caso 2: descripción que NO sea "estacional"
            "$regex": r"^estacional$",              # Regex que detecta "estacional"
            "$options": "i"                         # Ignora mayúsculas
        }}}
    ],
    MARK_RA: {"$ne": True}                          # Evita volver a aplicar el mismo update si ya se aplicó antes
}

pipeline_ra = [{                                   # Pipeline de actualización para “resto del año”
    "$set": {                                       # $set recalcula/crea campos en el documento
        # +2 mesas auxiliares resto año (campo nuevo)
        "mesas_aux_ra": {                           # Campo nuevo o existente: mesas auxiliares en resto del año
            "$add": [                               # Suma 2 al valor actual
                {"$convert": {                      # Convierte el valor actual a entero de forma segura
                    "input": "$mesas_aux_ra",       # Lee el valor actual del documento
                    "to": "int",                    # Lo convierte a int
                    "onError": 0,                   # Si hay error, usa 0
                    "onNull": 0                     # Si es null, usa 0
                }},
                2                                   # Incremento: +2 mesas auxiliares
            ]
        },
        # +8 sillas resto año
        "sillas_ra": {                              # Campo existente: sillas en resto del año
            "$add": [                               # Suma 8 al valor actual
                {"$convert": {                      # Convierte el valor actual a entero de forma segura
                    "input": "$sillas_ra",          # Lee el valor actual del documento
                    "to": "int",                    # Lo convierte a int
                    "onError": 0,                   # Si hay error, usa 0
                    "onNull": 0                     # Si es null, usa 0
                }},
                8                                   # Incremento: +8 sillas
            ]
        },
        MARK_RA: True,                              # Guarda la marca para que este update sea repetible (no se aplique dos veces)
        "_updated_at": now                          # Guarda timestamp del cambio
    }
}]

res_ra = col.update_many(filtro_ra, pipeline_ra)    # Ejecuta el update masivo para período RESTO DEL AÑO

print("✅ ESTACIONAL -> matched:", res_es.matched_count, "modified:", res_es.modified_count)  # Reporta cuántos encontró y cuántos cambió
print("✅ RESTO AÑO  -> matched:", res_ra.matched_count, "modified:", res_ra.modified_count)  # Reporta cuántos encontró y cuántos cambió

print("🔎 Sample ES:",                              # Imprime un ejemplo de documento actualizado en período estacional
      col.find_one(                                 # Busca 1 documento cualquiera que tenga la marca estacional aplicada
          {MARK_ES: True},                          # Filtro: ya actualizado por el paso estacional
          {"_id": 1, "id_local": 1, "id_periodo_terraza": 1,  # Proyección: campos que queremos ver
           "mesas_aux_es": 1, "sillas_es": 1, "inspeccionar": 1, MARK_ES: 1}
      ))

print("🔎 Sample RA:",                              # Imprime un ejemplo de documento actualizado en resto del año
      col.find_one(                                 # Busca 1 documento cualquiera que tenga la marca resto-año aplicada
          {MARK_RA: True},                          # Filtro: ya actualizado por el paso resto del año
          {"_id": 1, "id_local": 1, "id_periodo_terraza": 1,  # Proyección: campos que queremos ver
           "mesas_aux_ra": 1, "sillas_ra": 1, "inspeccionar": 1, MARK_RA: 1}
      ))

# Terrazas inspeccionables actualizadas en período ESTACIONAL
print(
    "ESTACIONAL (inspeccionar=True y marca ES):",
    col.count_documents({
        "inspeccionar": True,
        "_upd_63_es_v1": True
    })
)

# Terrazas inspeccionables actualizadas en RESTO DEL AÑO
print(
    "RESTO AÑO (inspeccionar=True y marca RA):",
    col.count_documents({
        "inspeccionar": True,
        "_upd_63_ra_v1": True
    })
)


✅ ESTACIONAL -> matched: 0 modified: 0
✅ RESTO AÑO  -> matched: 0 modified: 0
🔎 Sample ES: {'_id': 51, 'id_local': 270076142, 'id_periodo_terraza': 2, 'mesas_aux_es': 4, 'sillas_es': 51, 'inspeccionar': True, '_upd_63_es_v1': True}
🔎 Sample RA: {'_id': 41, 'id_local': 270495851, 'id_periodo_terraza': 1, 'mesas_aux_ra': 4, 'sillas_ra': 16, 'inspeccionar': True, '_upd_63_ra_v1': True}
ESTACIONAL (inspeccionar=True y marca ES): 179
RESTO AÑO (inspeccionar=True y marca RA): 618


**6.4** A las terrazas que no deban ser inspeccionadas, añadid el campo estado con el valor:  
1 si el número de sillas es menor que 10.  
2 si el número de sillas está entre 10 y 20.  
3 si cuenta con más de 20 sillas.


In [ ]:
from pymongo import MongoClient  # Importa el cliente para conectarse a MongoDB desde Python

client = MongoClient("mongodb://localhost:27017", serverSelectionTimeoutMS=5000)  # Crea conexión al MongoDB local (timeout 5s)
client.admin.command("ping")  # Verifica que el servidor responde (si falla, lanza excepción)
print("✅ Conectado")  # Mensaje de confirmación de conexión

col = client["Madrid"]["Terrazas"]  # Selecciona la base de datos "Madrid" y la colección "Terrazas"

# 1) Solo NO inspeccionar + SOLO período estacional
filtro = {
    "inspeccionar": False,  # Filtra únicamente las terrazas que NO deben inspeccionarse
    "$or": [  # Detecta el período estacional por id o por descripción (por si el dataset viene distinto)
        {"id_periodo_terraza": 2},  # Caso 1: id_periodo_terraza = 2 se interpreta como "Estacional"
        {"desc_periodo_terraza": {"$regex": r"^estacional$", "$options": "i"}}  # Caso 2: texto "Estacional" (sin importar mayúsculas)
    ]
}

# 2) Update con pipeline: estado según sillas_es (estacional)
update = [
    {
        "$set": {  # $set crea o actualiza campos en el documento
            "estado": {  # Campo nuevo "estado" que pide el enunciado
                "$let": {  # $let define una variable interna para reutilizarla
                    "vars": {  # Diccionario de variables internas del cálculo
                        "sillas": {  # Variable interna $$sillas: número de sillas convertido a entero
                            "$convert": {  # Convierte a entero de forma segura
                                "input": {"$ifNull": ["$sillas_es", 0]},  # Toma sillas_es; si es null/no existe usa 0
                                "to": "int",      # Tipo destino: entero
                                "onError": 0,     # Si sillas_es trae texto u otro error de conversión, usa 0
                                "onNull": 0       # Si es null, usa 0
                            }
                        }
                    },
                    "in": {  # Expresión final que usa la variable $$sillas para asignar "estado"
                        "$switch": {  # Switch/casos para decidir el valor del estado
                            "branches": [  # Lista de condiciones ordenadas
                                {"case": {"$lt": ["$$sillas", 10]}, "then": 1},   # Si sillas < 10  -> estado = 1
                                {"case": {"$lte": ["$$sillas", 20]}, "then": 2}   # Si sillas <= 20 -> estado = 2  (o sea 10..20)
                            ],
                            "default": 3  # Si no entra en los casos anteriores (sillas > 20) -> estado = 3
                        }
                    }
                }
            }
        }
    }
]

res = col.update_many(filtro, update)  # Ejecuta el update masivo usando pipeline 
print("✅ OK")  # Mensaje de confirmación de ejecución
print("Matched:", res.matched_count)  # Cantidad de documentos que cumplieron el filtro
print("Modified:", res.modified_count)  # Cantidad de documentos que realmente fueron modificados

# 3) Ver conteo por estado (solo no inspeccionar + estacional)
pipeline_check = [
    {"$match": filtro},  # Restringe el análisis al mismo subconjunto actualizado
    {"$group": {"_id": "$estado", "n": {"$sum": 1}}},  # Agrupa por estado y cuenta cuántos documentos hay en cada estado
    {"$sort": {"_id": 1}}  # Ordena por estado (1,2,3)
]

print("\n📌 Conteo por estado (inspeccionar=False, estacional):")  # Encabezado para mostrar resultados
for r in col.aggregate(pipeline_check):  # Ejecuta el pipeline de agregación y recorre los resultados
    print(r["_id"], "->", r["n"])  # Imprime: estado -> cantidad de documentos

# 4) Ejemplos
print("\n🔎 Ejemplos:")  # Encabezado para mostrar documentos ejemplo
for d in col.find(filtro, {"_id": 1, "sillas_es": 1, "estado": 1}).limit(5):  # Busca 5 docs del subconjunto y muestra campos relevantes
    print(d)  # Imprime cada documento de ejemplo

✅ Conectado
✅ OK
Matched: 568
Modified: 0

📌 Conteo por estado (inspeccionar=False, estacional):
1 -> 75
2 -> 223
3 -> 270

🔎 Ejemplos:
{'_id': 7, 'sillas_es': 0, 'estado': 1}
{'_id': 32, 'sillas_es': 20, 'estado': 2}
{'_id': 44, 'sillas_es': 24, 'estado': 3}
{'_id': 54, 'sillas_es': 24, 'estado': 3}
{'_id': 56, 'sillas_es': 16, 'estado': 2}


algunos registros sillas_ra viene con texto (por conversión/limpieza del CSV/JSON). No te rompe nada porque el cálculo usa sillas_es primero y además el $convert tiene onError: 0.

**6.5** De lunes a jueves, ningún local podrá cerrar más allá de las 00:00:00, actualizad el horario de cierre a esta nueva hora límite. 


In [661]:
from pymongo import MongoClient  # Cliente para conectarse a MongoDB desde Python

# =========================
# CONEXIÓN
# =========================
client = MongoClient("mongodb://localhost:27017", serverSelectionTimeoutMS=5000)  # Conecta al MongoDB local
client.admin.command("ping")  # Comprueba que el servidor responde
col = client["Madrid"]["Terrazas"]  # Selecciona la colección Madrid.Terrazas

# =========================
# FUNCIÓN: CAPAR HORA A 00:00:00 SI ES "DESPUÉS DE MEDIANOCHE"
# =========================
def cap_to_midnight_expr(field_name: str):
    """
    Devuelve una expresión MongoDB que:
    - lee un campo tipo "HH:MM:SS"
    - extrae HH (hora)
    - si HH está entre 1 y 11 => devuelve "00:00:00"
    - si no => deja el valor original
    """
    f = f"${field_name}"  # Referencia al campo en MongoDB (ej: "$hora_fin_LJ_es")

    return {
        "$let": {  # $let nos permite crear variables internas para reutilizarlas
            "vars": {
                "t": {"$ifNull": [f, ""]},  # $$t = valor del campo o "" si no existe/null

                # $$h = hora (HH) convertida a entero; si falla => None
                "h": {
                    "$convert": {
                        "input": {
                            "$arrayElemAt": [                      # toma el primer elemento tras split
                                {"$split": [{"$ifNull": [f, ""]}, ":"]},  # split "HH:MM:SS" por ":"
                                0
                            ]
                        },
                        "to": "int",     # convierte "HH" a int
                        "onError": None, # si no puede convertir, usa None
                        "onNull": None   # si es null, usa None
                    }
                }
            },

            "in": {
                "$cond": [
                    # CONDICIÓN: existe hora válida y está en rango 01..11 (después de medianoche)
                    {
                        "$and": [
                            {"$ne": ["$$t", ""]},                 # el string no está vacío
                            {"$ne": ["$$h", None]},               # la hora se pudo parsear
                            {"$gt": ["$$h", 0]},                  # HH > 0  (01..)
                            {"$lt": ["$$h", 12]},                 # HH < 12 (..11)
                            {"$ne": ["$$t", "00:00:00"]}          # si ya es medianoche, no tocar
                        ]
                    },

                    "00:00:00",  # SI cumple => capamos a medianoche
                    f            # SI no cumple => dejamos el valor original
                ]
            }
        }
    }

# =========================
# FILTRO: PERÍODO ESTACIONAL (según el enunciado)
# =========================
filter_q = {"id_periodo_terraza": 2}  # Solo periodo estacional

# =========================
# UPDATE MASIVO (LUNES A JUEVES)
# =========================
res = col.update_many(
    filter_q,  # Aplica a documentos estacionales
    [
        {"$set": {
            # Ajusta el cierre de lunes a jueves (campo estacional)
            "hora_fin_LJ_es": cap_to_midnight_expr("hora_fin_LJ_es"),

            # Ajusta el cierre de lunes a jueves (campo resto año)
            # Nota: el inciso dice actualizar la hora límite, y como tu dataset tiene LJ_es y LJ_ra,
            # lo capamos en ambos para mantener coherencia.
            "hora_fin_LJ_ra": cap_to_midnight_expr("hora_fin_LJ_ra"),

            # Marca para auditoría / repetibilidad (opcional)
            "_upd_hora_fin_LJ_v1": True
        }}
    ]
)

# =========================
# REPORTES
# =========================
print("✅ OK")  # Confirmación
print("Matched:", res.matched_count)   # Documentos que cumplían el filtro
print("Modified:", res.modified_count) # Documentos efectivamente modificados

# =========================
# EJEMPLO PARA VERIFICAR
# =========================
sample = col.find_one(
    {
        "id_periodo_terraza": 2,  # estacional
        "$or": [
            {"hora_fin_LJ_es": "00:00:00"},  # alguno quedó en medianoche
            {"hora_fin_LJ_ra": "00:00:00"}
        ]
    },
    {
        "_id": 1,
        "hora_fin_LJ_es": 1,
        "hora_fin_LJ_ra": 1,
        "desc_periodo_terraza": 1
    }
)

print("Sample:", sample)  # Muestra un documento ya actualizado

✅ OK
Matched: 888
Modified: 0
Sample: {'_id': 7, 'desc_periodo_terraza': 'Estacional', 'hora_fin_LJ_es': '00:00:00', 'hora_fin_LJ_ra': ''}


**6.6** De viernes a sábado, los locales que cierren a las 2:30:00 ahora tendrán que hacerlo a las 2:00:00. 


In [716]:
from pymongo import MongoClient  # Importa el cliente de MongoDB para poder conectarnos desde Python

# 1) Conexión
client = MongoClient("mongodb://localhost:27017", serverSelectionTimeoutMS=5000)  # Crea conexión al MongoDB local (timeout 5s)
client.admin.command("ping")  # Verifica que el servidor MongoDB responde correctamente
col = client["Madrid"]["Terrazas"]  # Selecciona la base de datos "Madrid" y la colección "Terrazas"

# 2) Filtro base: SOLO período estacional
# Nota del enunciado: si se actualizan campos *_es o *_ra, el update debe hacerse sobre datos del período estacional
BASE = {"id_periodo_terraza": 2}  # En el dataset, id_periodo_terraza=2 corresponde a "Estacional"

# 3) Normaliza posibles formatos (con y sin 0)
OLD_TIMES = ["02:30:00", "2:30:00"]  # Posibles valores antiguos del cierre (con cero a la izquierda o sin él)
NEW_TIME  = "02:00:00"               # Nuevo valor de cierre requerido por el inciso 6.6 (2:00:00)

# 4) Update VS estacional (_es y _ra)
# Inciso 6.6: De viernes a sábado (VS), si cierran a las 2:30:00 ahora deben cerrar a las 2:00:00

# Actualiza el campo de cierre VS en período estacional para la parte *_es
res_es = col.update_many(
    {**BASE, "hora_fin_VS_es": {"$in": OLD_TIMES}},  # Filtra docs estacionales cuya hora_fin_VS_es sea 02:30:00 o 2:30:00
    {"$set": {"hora_fin_VS_es": NEW_TIME}}           # Reemplaza el valor por 02:00:00
)

# Actualiza el campo de cierre VS en período estacional para la parte *_ra
res_ra = col.update_many(
    {**BASE, "hora_fin_VS_ra": {"$in": OLD_TIMES}},  # Filtra docs estacionales cuya hora_fin_VS_ra sea 02:30:00 o 2:30:00
    {"$set": {"hora_fin_VS_ra": NEW_TIME}}           # Reemplaza el valor por 02:00:00
)

print("✅ Update VS (estacional)")  # Mensaje de confirmación de ejecución
print("ES matched/mod:", res_es.matched_count, res_es.modified_count)  # matched=docs encontrados, modified=docs realmente cambiados
print("RA matched/mod:", res_ra.matched_count, res_ra.modified_count)  # matched=docs encontrados, modified=docs realmente cambiados

# 5) Verificación (debe dar 0 si ya quedó aplicado)
# Si el update se aplicó correctamente, ya no debería quedar ningún documento con 02:30:00 en esos campos.
left_es = col.count_documents({**BASE, "hora_fin_VS_es": {"$in": OLD_TIMES}})  # Cuenta cuántos siguen con 02:30:00 (ES)
left_ra = col.count_documents({**BASE, "hora_fin_VS_ra": {"$in": OLD_TIMES}})  # Cuenta cuántos siguen con 02:30:00 (RA)
print("\nQuedan cierres 02:30 VS (estacional) en ES:", left_es)  # Idealmente 0
print("Quedan cierres 02:30 VS (estacional) en RA:", left_ra)     # Idealmente 0

# Muestra algunos documentos que ya tengan el nuevo horario 02:00:00 .

print("\nEjemplos ES:")
for d in col.find(
    {**BASE, "hora_fin_VS_es": NEW_TIME},            # Filtra docs estacionales ya actualizados en hora_fin_VS_es
    {"_id": 1, "hora_fin_VS_es": 1}                  # Proyección: muestra solo _id y el campo actualizado
).limit(5):                                          # Muestra máximo 5 ejemplos
    print(d)                                         # Imprime cada ejemplo

print("\nEjemplos RA:")
for d in col.find(
    {**BASE, "hora_fin_VS_ra": NEW_TIME},            # Filtra docs estacionales ya actualizados en hora_fin_VS_ra
    {"_id": 1, "hora_fin_VS_ra": 1}                  # Proyección: muestra solo _id y el campo actualizado
).limit(5):                                          # Muestra máximo 5 ejemplos
    print(d)                                         # Imprime cada ejemplo

✅ Update VS (estacional)
ES matched/mod: 0 0
RA matched/mod: 0 0

Quedan cierres 02:30 VS (estacional) en ES: 0
Quedan cierres 02:30 VS (estacional) en RA: 0

Ejemplos ES:
{'_id': 44, 'hora_fin_VS_es': '02:00:00'}
{'_id': 72, 'hora_fin_VS_es': '02:00:00'}
{'_id': 136, 'hora_fin_VS_es': '02:00:00'}
{'_id': 154, 'hora_fin_VS_es': '02:00:00'}
{'_id': 156, 'hora_fin_VS_es': '02:00:00'}

Ejemplos RA:


El cambio “de viernes a sábado: 02:30 → 02:00” solo aplicaba a hora_fin_VS_es (520 casos). En hora_fin_VS_ra no se aplicó porque no existían registros con 02:30 en ese campo (0 casos).

**6.7** A todos los locales que estén sobre la calle Alcalá en Madrid se les debe inspeccionar. 

In [ ]:
from pymongo import MongoClient  # Importa el cliente oficial de MongoDB para Python (pymongo)

# =========================
# 1) Conexión a MongoDB
# =========================
client = MongoClient("mongodb://localhost:27017", serverSelectionTimeoutMS=5000)  # Crea conexión al servidor MongoDB local (timeout 5s)
client.admin.command("ping")  # Envía un "ping" al servidor para confirmar que está accesible
print("✅ Conectado")  # Mensaje de confirmación si la conexión fue exitosa

col = client["Madrid"]["Terrazas"]  # Selecciona la base de datos "Madrid" y la colección "Terrazas"

# =========================
# 2) Detectar el campo de calle/dirección 
# =========================
candidatos = [  # Lista de nombres de campos típicos donde podría venir el nombre de la calle/vía
    "DESC_NOMBRE", "desc_nombre",
    "nombre_via", "nombre_calle", "calle",
    "desc_via", "via", "direccion", "domicilio"
]

campos_existentes = []  # Aquí guardaremos los campos que realmente existan en los documentos de la colección

for f in candidatos:  # Recorre cada nombre de campo candidato
    # Busca si existe al menos un documento que tenga ese campo con un valor no vacío
    if col.find_one({f: {"$exists": True, "$ne": None, "$ne": ""}}, {f: 1}):
        campos_existentes.append(f)  # Si existe, lo añadimos a la lista de campos válidos

if not campos_existentes:  # Si no encontramos ningún campo de calle/dirección
    sample = col.find_one()  # Tomamos un documento de ejemplo para mostrar sus claves disponibles
    raise ValueError(  # Lanzamos error para detener el script y avisar al usuario/profesor
        "No encontré ningún campo típico de nombre de calle.\n"
        f"Claves de un doc ejemplo: {list(sample.keys()) if sample else 'colección vacía'}"
    )

print("🧭 Campos candidatos encontrados:", campos_existentes)  # Muestra qué campos se detectaron como "calle"

# =========================
# 3) Filtro: documentos cuya calle contenga Alcalá/Alcala
# =========================
regex_alcala = r"alcal(?:a|á)"  # Expresión regular: acepta "Alcala" y "Alcalá" (con o sin acento)

filtro_alcala = {  # Crea un filtro MongoDB que busca "Alcalá/Alcala" en cualquiera de los campos detectados
    "$or": [{f: {"$regex": regex_alcala, "$options": "i"}} for f in campos_existentes]  # "$options": "i" = case-insensitive
}

# (opcional) hacerlo repetible: evita re-aplicar si ya estaba marcado por este inciso
MARK = "_upd_67_alcala_v1"  # Campo "marca" para auditoría: indica que el inciso 6.7 ya fue aplicado
filtro_alcala_repetible = {**filtro_alcala, MARK: {"$ne": True}}  # Solo actualiza documentos que NO tengan la marca en True

# =========================
# 4) Evidencia ANTES del update
# =========================
antes_total = col.count_documents(filtro_alcala)  # Cuenta cuántos documentos están en Alcalá (según el filtro)
antes_true  = col.count_documents({**filtro_alcala, "inspeccionar": True})  # De esos, cuenta cuántos ya tienen inspeccionar=True
antes_false = col.count_documents({**filtro_alcala, "inspeccionar": {"$ne": True}})  # Cuenta los que NO tienen inspeccionar=True

print("\n📌 Antes del update (Alcalá):")  # Encabezado para la evidencia previa
print("Total en Alcalá:", antes_total)  # Total de documentos Alcalá antes del update
print("inspeccionar=True:", antes_true)  # Cuántos ya estaban marcados para inspección
print("inspeccionar!=True:", antes_false)  # Cuántos faltaban por marcar

# =========================
# 5) Update: todos los de Alcalá deben inspeccionarse
# =========================
res = col.update_many(  # Ejecuta una actualización masiva sobre todos los documentos que cumplen el filtro
    filtro_alcala_repetible,  # Filtro repetible (solo los que aún no tienen la marca)
    {"$set": {"inspeccionar": True, MARK: True}}  # Asigna inspeccionar=True y añade la marca del inciso 6.7
)

print("\n✅ Update 6.7 (Alcalá)")  # Encabezado de confirmación
print("Matched:", res.matched_count, "| Modified:", res.modified_count)  # matched=coinciden con filtro, modified=realmente cambiados

# =========================
# 6) Evidencia DESPUÉS del update
# =========================
desp_total = col.count_documents(filtro_alcala)  # Recuenta documentos Alcalá (debería ser el mismo total)
desp_true  = col.count_documents({**filtro_alcala, "inspeccionar": True})  # Recuenta cuántos quedaron con inspeccionar=True
desp_false = col.count_documents({**filtro_alcala, "inspeccionar": {"$ne": True}})  # Recuenta los que aún no están en True

print("\n📌 Después del update (Alcalá):")  # Encabezado para la evidencia posterior
print("Total en Alcalá:", desp_total)  # Total de documentos Alcalá después del update
print("inspeccionar=True:", desp_true)  # Cuántos quedaron marcados
print("inspeccionar!=True:", desp_false)  # Cuántos quedaron sin marcar (idealmente 0)

# =========================
# 7) Muestra ejemplos reales (prueba)
# =========================
proj = {"_id": 1, "id_local": 1, "inspeccionar": 1, MARK: 1}  # Proyección: solo devuelve estos campos para no imprimir documentos completos

for f in campos_existentes:  # Añade también los campos de calle detectados, para ver en cuál aparece "Alcalá"
    proj[f] = 1

print("\n🔎 Ejemplos (Alcalá) ya marcados:")  # Encabezado de ejemplos
for d in col.find({**filtro_alcala, "inspeccionar": True}, proj).limit(5):  # Busca 5 docs de Alcalá ya marcados para inspección
    print(d)  # Imprime cada documento ejemplo (con los campos seleccionados)

✅ Conectado
🧭 Campos candidatos encontrados: ['DESC_NOMBRE']

📌 Antes del update (Alcalá):
Total en Alcalá: 50
inspeccionar=True: 50
inspeccionar!=True: 0

✅ Update 6.7 (Alcalá)
Matched: 0 | Modified: 0

📌 Después del update (Alcalá):
Total en Alcalá: 50
inspeccionar=True: 50
inspeccionar!=True: 0

🔎 Ejemplos (Alcalá) ya marcados:
{'_id': 41, 'id_local': 270495851, 'DESC_NOMBRE': 'ALCALA', 'inspeccionar': True, '_upd_67_alcala_v1': True}
{'_id': 51, 'id_local': 270076142, 'DESC_NOMBRE': 'ALCALA', 'inspeccionar': True, '_upd_67_alcala_v1': True}
{'_id': 52, 'id_local': 270071068, 'DESC_NOMBRE': 'ALCALA', 'inspeccionar': True, '_upd_67_alcala_v1': True}
{'_id': 58, 'id_local': 270075832, 'DESC_NOMBRE': 'ALCALA', 'inspeccionar': True, '_upd_67_alcala_v1': True}
{'_id': 59, 'id_local': 40001037, 'DESC_NOMBRE': 'ALCALA', 'inspeccionar': True, '_upd_67_alcala_v1': True}


**6.8** A todos los locales con terrazas abiertas, añadid un campo revision cuyo valor sea un documento con la siguiente información: {prox_inspeccion: 10, puntuacion: 80, comentario: “separar las mesas”}

In [801]:
from pymongo import MongoClient  # Importa el cliente oficial de MongoDB para Python (PyMongo)

# Crea un cliente apuntando a MongoDB en localhost (puerto 27017) y limita el tiempo de espera
client = MongoClient("mongodb://localhost:27017", serverSelectionTimeoutMS=5000)

# Verifica conectividad haciendo un "ping" al servidor (si falla, lanza excepción)
client.admin.command("ping")

# Selecciona la colección Madrid.Terrazas (si no existe, Mongo la crea cuando insertas/actualizas)
col = client["Madrid"]["Terrazas"]

# -----------------------------
# 1) Filtro: "terrazas abiertas"
# -----------------------------
# Consideramos "abiertas" si:
#  - desc_situacion_terraza == "Abierta" (ignorando mayúsculas/minúsculas), o
#  - id_situacion_terraza == 1 (según la codificación observada en el dataset)
f_abiertas = {
    "$or": [
        {"desc_situacion_terraza": {"$regex": r"^abierta$", "$options": "i"}},  # "Abierta" (case-insensitive)
        {"id_situacion_terraza": 1},                                           # código numérico de "Abierta"
    ]
}

# -----------------------------
# 2) Documento embebido "revision"
# -----------------------------
# Este será el valor del nuevo campo "revision" (un documento dentro del documento principal)
revision_doc = {
    "prox_inspeccion": 10,          # días hasta la próxima inspección (según enunciado)
    "puntuacion": 80,               # puntuación (según enunciado)
    "comentario": "separar las mesas"  # comentario (según enunciado)
}

# -----------------------------
# 3) Marca para hacer el update repetible (idempotente)
# -----------------------------
# Si un documento ya tiene MARK=True, este script no lo vuelve a modificar (evita "re-aplicar")
MARK = "_upd_68_revision_v1"

# Construye un filtro "repetible":
#  - debe cumplir que la terraza esté abierta (f_abiertas)
#  - y además que NO tenga la marca MARK=True todavía
filtro_repetible = {**f_abiertas, MARK: {"$ne": True}}

# -----------------------------
# 4) Update masivo
# -----------------------------
# Aplica a todos los docs que cumplan el filtro:
#  - crea/actualiza el campo "revision" con el documento revision_doc
#  - añade la marca MARK=True para saber que este inciso ya fue aplicado
res = col.update_many(
    filtro_repetible,
    {"$set": {"revision": revision_doc, MARK: True}}
)

# Imprime resumen del resultado del update:
# - matched_count: cuántos documentos coincidieron con el filtro
# - modified_count: cuántos documentos realmente cambiaron (se actualizaron)
print("✅ Update revision (abiertas)")
print("Matched:", res.matched_count, "| Modified:", res.modified_count)

# -----------------------------
# 5) Evidencia / verificación rápida
# -----------------------------
# Busca 1 documento que sea "abierto" y muestra campos clave para comprobar que se añadió "revision"
sample = col.find_one(
    f_abiertas,
    {
        "_id": 1,
        "desc_situacion_terraza": 1,
        "id_situacion_terraza": 1,
        "revision": 1,
        MARK: 1
    }
)

# Muestra un ejemplo real para evidencia en el reporte
print("Sample:", sample)

f_abiertas = {
    "$or": [
        {"desc_situacion_terraza": {"$regex": r"^abierta$", "$options": "i"}},
        {"id_situacion_terraza": 1},
    ]
}

revision_doc = {"prox_inspeccion": 10, "puntuacion": 80, "comentario": "separar las mesas"}

pendientes = col.count_documents({**f_abiertas, "revision": {"$ne": revision_doc}})
print("Pendientes (abiertas sin revision correcta):", pendientes)

for d in col.find(f_abiertas, {"_id": 1, "desc_situacion_terraza": 1, "id_situacion_terraza": 1, "revision": 1}).limit(3):
    print(d)



✅ Update revision (abiertas)
Matched: 0 | Modified: 0
Sample: {'_id': 7, 'id_situacion_terraza': 1, 'desc_situacion_terraza': 'Abierta', 'revision': {'prox_inspeccion': 10, 'puntuacion': 80, 'comentario': 'separar las mesas'}, '_upd_68_revision_v1': True}
Pendientes (abiertas sin revision correcta): 0
{'_id': 7, 'id_situacion_terraza': 1, 'desc_situacion_terraza': 'Abierta', 'revision': {'prox_inspeccion': 10, 'puntuacion': 80, 'comentario': 'separar las mesas'}}
{'_id': 32, 'id_situacion_terraza': 1, 'desc_situacion_terraza': 'Abierta', 'revision': {'prox_inspeccion': 10, 'puntuacion': 80, 'comentario': 'separar las mesas'}}
{'_id': 40, 'id_situacion_terraza': 1, 'desc_situacion_terraza': 'Abierta', 'revision': {'prox_inspeccion': 10, 'puntuacion': 80, 'comentario': 'separar las mesas'}}


**6.9** Cread una nueva colección llamada Zona1, con todos los locales que pertenezcan al distrito de Villaverde.

In [806]:
from pymongo import MongoClient  # Importa el cliente oficial para conectarse a MongoDB desde Python

client = MongoClient("mongodb://localhost:27017", serverSelectionTimeoutMS=5000)  # Crea conexión al servidor MongoDB local (timeout 5s)
client.admin.command("ping")  # Hace un "ping" para confirmar que el servidor responde (si falla, lanza error)

db = client["Madrid"]         # Selecciona (o crea si no existe) la base de datos llamada "Madrid"
src = db["Terrazas"]          # Selecciona la colección fuente "Terrazas" dentro de la BD Madrid

# 1) Detectar cómo se llama la columna del distrito (por si cambia el nombre)
sample = src.find_one({}, {"_id": 0}) or {}  # Lee 1 documento de ejemplo (sin _id) para ver qué campos existen; si no hay docs, usa {}
candidates = ["desc_distrito_local", "DESC_DISTRITO_LOCAL", "desc_distrito", "DESC_DISTRITO"]  # Posibles nombres del campo distrito
field = next((f for f in candidates if f in sample), None)  # Elige el primer candidato que exista en el documento de ejemplo
if not field:  # Si ninguno de los candidatos está presente...
    raise ValueError(  # ...lanza un error explicando que no encontró el campo de distrito
        f"No encuentro un campo de distrito. Campos disponibles: {list(sample.keys())[:40]}"
    )

# 2) Crear/reemplazar la colección Zona1 con $out (repetible)
pipeline = [  # Define un pipeline de agregación (pasos) para filtrar y escribir a otra colección
    {"$match": {field: {"$regex": r"^\s*VILLAVERDE\s*$", "$options": "i"}}},  # Filtra docs cuyo distrito sea "VILLAVERDE" (ignora mayús/minús y espacios)
    {"$out": "Zona1"}  # Escribe el resultado en la colección "Zona1" (la crea o la reemplaza completamente)
]
list(src.aggregate(pipeline, allowDiskUse=True))  # Ejecuta el pipeline (list() fuerza la ejecución); allowDiskUse permite usar disco si hace falta

# 3) Verificación
zona1 = db["Zona1"]  # Selecciona la colección "Zona1" ya creada/reemplazada
print("✅ Zona1 creada. Documentos:", zona1.count_documents({}))  # Imprime cuántos documentos hay en Zona1
print(  # Imprime un documento de ejemplo (solo algunos campos) para comprobar que el filtrado fue correcto
    "Ejemplo:",
    zona1.find_one({}, {"_id": 1, field: 1, "id_local": 1, "Nombre": 1, "DESC_NOMBRE": 1})
)


✅ Zona1 creada. Documentos: 119
Ejemplo: {'_id': 54, 'id_local': 280062208, 'desc_distrito_local': 'VILLAVERDE', 'Nombre': 'BAR LAS TORRES', 'DESC_NOMBRE': 'MONCADA'}


**6.10** Cread una nueva colección llamada Zona2, con todos los locales que pertenezcan al distrito de Salamanca y barrio Castellana.

In [813]:
from pymongo import MongoClient  # Importa el cliente oficial de MongoDB para conectarnos y ejecutar queries

# =========================
# 1) Conexión a MongoDB
# =========================
client = MongoClient("mongodb://localhost:27017", serverSelectionTimeoutMS=5000)  # Conecta al servidor MongoDB local (puerto 27017) con timeout
client.admin.command("ping")  # Verifica que el servidor responde (si falla, lanza error)
print("✅ Conectado")  # Mensaje de confirmación de conexión exitosa

# =========================
# 2) Selección de base de datos y colección fuente
# =========================
db = client["Madrid"]          # Selecciona (o crea si no existe) la base de datos llamada "Madrid"
col = db["Terrazas"]           # Selecciona la colección "Terrazas" (donde están los documentos originales)

# =========================
# 3) Definir filtro para Salamanca + Castellana (tolerante a espacios y mayúsculas)
# =========================
filtro = {
    "desc_distrito_local": {"$regex": r"^\s*SALAMANCA\s*$", "$options": "i"},  # Distrito exactamente "SALAMANCA" ignorando espacios y mayúsculas/minúsculas
    "desc_barrio_local":   {"$regex": r"^\s*CASTELLANA\s*$", "$options": "i"}, # Barrio exactamente "CASTELLANA" ignorando espacios y mayúsculas/minúsculas
}

# =========================
# 4) Pipeline de agregación para crear Zona2 (REPETIBLE)
# =========================
pipeline = [
    {"$match": filtro},  # Paso 1: filtra documentos de Terrazas que cumplen el filtro (Salamanca + Castellana)
    {"$out": "Zona2"}    # Paso 2: escribe el resultado en una nueva colección llamada "Zona2" (la reemplaza si ya existe)
]

# Ejecuta el pipeline en MongoDB
col.aggregate(pipeline, allowDiskUse=True)  # allowDiskUse permite usar disco si el pipeline usa mucha memoria

# =========================
# 5) Verificación: conteo de documentos creados en Zona2
# =========================
zona2 = db["Zona2"]                     # Referencia a la colección recién creada/actualizada
n = zona2.count_documents({})           # Cuenta cuántos documentos quedaron en Zona2
print(f"✅ Zona2 creada/actualizada. Documentos: {n}")  # Imprime evidencia del tamaño de Zona2

# =========================
# 6) Verificación cruzada: conteo en la colección fuente con el mismo filtro
# =========================
print("🔎 Verificación (Terrazas con ese filtro):", col.count_documents(filtro))  # Debe coincidir con el conteo de Zona2

# =========================
# 7) Mostrar un ejemplo de documento en Zona2 (para evidencia)
# =========================
print(
    "📌 Ejemplo:",
    zona2.find_one(
        {},  # Sin filtro: toma cualquier documento
        {    # Proyección: solo muestra campos relevantes para el reporte
            "_id": 1,
            "id_local": 1,
            "desc_distrito_local": 1,
            "desc_barrio_local": 1,
            "Nombre": 1,
            "DESC_NOMBRE": 1
        }
    )
)

# =========================
# 8) Extra: medir si hay múltiples docs por local (porque Terrazas puede tener varios registros por id_local)
# =========================
print("Docs en Zona2:", zona2.count_documents({}))                # Total de documentos en Zona2
print("Locales únicos en Zona2:", len(zona2.distinct("id_local")))# Cantidad de id_local distintos (locales únicos)


✅ Conectado
✅ Zona2 creada/actualizada. Documentos: 44
🔎 Verificación (Terrazas con ese filtro): 44
📌 Ejemplo: {'_id': 519, 'id_local': 290000070, 'desc_distrito_local': 'SALAMANCA', 'desc_barrio_local': 'CASTELLANA', 'Nombre': 'MAMA NO LO SABE', 'DESC_NOMBRE': 'CASTELLO'}
Docs en Zona2: 44
Locales únicos en Zona2: 44


**7. Neo4j**  

En Neo4J, proponed un modelo de datos donde sea posible visualizar los locales de cada barrio y los tipos de terrazas que estos tienen. Indicad los nodos y sus relaciones, cread una única consulta que visualice el grafo total (incluidas las relaciones). Añadid tanto a nodos como a relaciones sus atributos, para ello elegid los campos que más convengan a vuestro modelo.

**7.2** Una breve descripción del modelo propuesto.

Este modelo representa la jerarquía territorial Distrito → Barrio → Local  y conecta cada  Local  con su  Terraza. Además, se crea un nodo TipoTerraza para clasificar las terrazas por criterios relevantes del dataset (por ejemplo, ubicación y período), permitiendo visualizar rápidamente qué tipos de terrazas predominan en cada barrio y qué locales las poseen.
Las relaciones incluyen atributos de trazabilidad como source y loaded_at para documentar el origen y momento de carga, facilitando auditoría y recargas repetibles con MERGE.

**7.3** La descripción del proceso utilizado para cargar los datos en Neo4J a partir de los datos en MongoDB:

Primero, exportamos la base de datos de MongoDB a un CSV para usar Neo4j.

In [817]:
from pymongo import MongoClient
from pathlib import Path
import csv

client = MongoClient("mongodb://localhost:27017", serverSelectionTimeoutMS=5000)
client.admin.command("ping")

col = client["Madrid"]["Terrazas"]

OUT = Path(r"C:\Users\getu9\Desktop\MaestriaCienciasdeDatos\MaestriaUnirDatos\terrazas_neo4j.csv")

fields = [
    "_id",
    "id_distrito_local","desc_distrito_local",
    "id_barrio_local","desc_barrio_local",
    "id_local","Nombre",
    "desc_ubicacion_terraza","cal_terraza","desc_periodo_terraza",
    "id_situacion_terraza","desc_situacion_terraza",
    "mesas_es","sillas_es","mesas_ra","sillas_ra",
    "inspeccionar","estado"
]

with OUT.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fields, delimiter=";")
    w.writeheader()

    for d in col.find({}, {k: 1 for k in fields}):
        row = {}
        for k in fields:
            v = d.get(k, "")
            row[k] = "" if v is None else str(v).strip()
        w.writerow(row)

print("✅ CSV creado:", OUT)


✅ CSV creado: C:\Users\getu9\Desktop\MaestriaCienciasdeDatos\MaestriaUnirDatos\terrazas_neo4j.csv
